In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import datetime
import os
import json
import requests
from bs4 import BeautifulSoup
import shutil
import time

In [2]:
# first thing first  we will use movie lens data set to get movie ids
# then we will use tmdb api to get movie data for each movie id
# we will get data like movie title , genres , overview , release date ,poster path etc
ids= pd.read_csv(r'E:\ml-32m/links.csv')

In [3]:
ids.head()

,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0
3,4,114885,31357.0
4,5,113041,11862.0


In [4]:
TMDB_API_KEY = "dcbbf062e91bfc7ee36c9aa743dbba09"
TMDB_BASE_URL = "https://api.themoviedb.org/3"
TMDB_IMAGE_BASE_URL = "https://image.tmdb.org/t/p/w500"

In [6]:
BACKUP_FILE = "tmdb_movies_backup.csv"
def get_tmdb_movie_data(tmdb_id):
    url = f"{TMDB_BASE_URL}/movie/{tmdb_id}"
    params = {"api_key": TMDB_API_KEY}
    response = requests.get(url, params=params)
    if response.status_code != 200:
        return None
    data = response.json()
    
    keywords_url = f"{TMDB_BASE_URL}/movie/{tmdb_id}/keywords"
    keywords_response = requests.get(keywords_url, params=params)
    keywords = keywords_response.json().get("keywords", []) if keywords_response.status_code == 200 else []
    
    credits_url = f"{TMDB_BASE_URL}/movie/{tmdb_id}/credits"
    credits_response = requests.get(credits_url, params=params)
    credits = credits_response.json() if credits_response.status_code == 200 else {}
    cast = [member["name"] for member in credits.get("cast", [])[:5]]
    director = next((member["name"] for member in credits.get("crew", []) if member["job"] == "Director"), None)
    
    result = {
        "movie_id": tmdb_id,
        "title": data.get("title"),
        "overview": data.get("overview"),
        "genres": [genre["name"] for genre in data.get("genres", [])],
        "keywords": [kw["name"] for kw in keywords],
        "cast": cast,
        "director": director,
        "release_date": data.get("release_date"),
        "runtime": data.get("runtime"),
        "popularity": data.get("popularity"),
        "poster_url": TMDB_IMAGE_BASE_URL + data["poster_path"] if data.get("poster_path") else None
    }
    return result

# Load backup if exists
if os.path.exists(BACKUP_FILE):
   movies_df = pd.read_csv(BACKUP_FILE)
   done_ids = set(movies_df["movie_id"].astype(str))
   print(f"Resuming from backup, {len(done_ids)} movies already processed.")
else:
    movies_df = pd.DataFrame()
    done_ids = set()

results = []
start_idx = len(done_ids)
tmdb_ids = ids["tmdbId"].dropna().astype(int).astype(str).tolist()

for idx, tmdb_id in enumerate(tmdb_ids[start_idx:], start=start_idx):
    if tmdb_id in done_ids:
        continue
    movie_data = get_tmdb_movie_data(tmdb_id)
    if movie_data:
        results.append(movie_data)
    time.sleep(0.25)  # Respect TMDB rate limits
    # Save backup every 10 movies
    if (idx + 1) % 10 == 0 or (idx + 1) == len(tmdb_ids):
        temp_df = pd.DataFrame(results)
        movies_df = pd.concat([movies_df, temp_df], ignore_index=True)
        movies_df.to_csv(BACKUP_FILE, index=False)
        results = []
        actual_saved_count = len(set(movies_df["movie_id"].astype(str)))
        print(f"Saved backup at {actual_saved_count} movies.")

print("Done! All movies processed and saved.")

Resuming from backup, 86275 movies already processed.


KeyboardInterrupt: 